# Bankruptcy Prediction
Name: Ruilin Yang

# Problem description

You are to predict whether a company will go bankrupt in the following
year, based on financial attributes of the company.

Perhaps you are contemplating lending money to a company, and need to know
whether the company is in near-term danger of not being able to repay.


## Learning objectives

- Demonstrate mastery on solving a classification problem and presenting
your end-to-end approach in a notebook.
- We will make suggestions for ways to approach the problem
    - But there will be little explicit direction for this task.
- It is meant to be analogous to a pre-interview task that a potential
employer might assign to verify your skill.


# Before we start: why do companies go bankrupt?

At the simplest level, a company runs into trouble when the money coming
in isn't enough to cover the money it must pay out.

- **Money coming in**: revenue from sales, and the profit left over after
costs.
- **Money going out**: the costs of doing business, and -- critically --
debts coming due, often with interest attached.
- **The cushion in between**: cash and other liquid resources on hand, and
the equity buffer built up by owners over time.

A company heading toward bankruptcy typically shows some mix of: shrinking
or negative profit, growing obligations, and a thinning cushion. The
dataset gives you dozens of financial attributes that each touch on one
piece of this picture. Part of your job is figuring out which pieces
matter most, and how they combine.


# A light glossary of financial terms

You don't need an accounting background to do well on this assignment --
recognizing patterns in the data matters more than knowing every term by
heart. But since the attributes use real financial-statement language,
here's a plain-English reference.

## Revenue & Income (money the company earns)
- **sales** -- total revenue generated from selling products/services
during the year.
- **Gross Profit** -- what's left from sales after subtracting the direct
cost of producing what was sold.
- **Net Profit** -- the bottom line: what's left after *all* expenses,
interest, and taxes are paid.
- **Operating Profit** -- profit from normal day-to-day business, before
financing costs or one-off items.
- **Profit on Sales** -- profit tied specifically to sales activity.
- **EBIT** -- profit before interest and taxes are subtracted, a way to
look at the core business regardless of how it's financed or taxed.
- **EBITDA** -- like EBIT, but also before non-cash accounting charges like
depreciation -- often used as a rough stand-in for cash the business
generates.
- A few other columns are close *flavors of* gross profit -- for example,
with interest or depreciation added back in, or totaled over three years
instead of one. Treat these as closely related variations; the specific
adjustment matters less than recognizing they're all measuring
profitability.

## Costs & Obligations (money the company owes or spends)
- **total_liabilities** -- everything the company owes, both short- and
long-term.
- **short_term_liabilities** -- obligations due within the coming year
(some formulas below call this "current liabilities" -- same thing).
- **Operating Expenses** -- the everyday costs of running the business.
- **Cost of Products Sold** -- the direct cost of producing or acquiring
the goods a company actually sold (sometimes abbreviated COGS).
- **Financial Expenses** -- the cost of financing, mainly interest paid
on borrowed money.
- **Liabilities Net of Cash** -- what the company owes, minus the cash it
has on hand to help pay it off.

## Ownership & Capital (money invested by owners, not borrowed)
- **Owner's Equity** -- the value of the company belonging to its owners
after all debts are subtracted.
- **Retained Earnings** -- accumulated profit kept in the business over
time rather than paid out to owners.
- A couple of other columns are close *flavors of* equity -- one reflects
equity built up from operations rather than what owners first put in,
another (sometimes called "constant capital") combines equity with
long-term borrowing into a single "funds we don't need to repay soon"
figure. These are related capital measures, not fundamentally different
ideas.

## Resources & Assets (what the company has on hand or owns longer-term)
- **total_assets** -- the total value of everything the company owns.
- **Current Assets** -- assets expected to be converted to cash or used up
within a year (things like inventory, receivables, and cash itself).
- **Fixed Assets** -- longer-term physical resources, like property and
equipment, not expected to convert to cash within a year.
- **Inventory** -- the value of goods held for sale.
- **Accounts Receivable** -- money customers owe the company for goods or
services already delivered.
- **Working Capital** -- the cushion left over after covering near-term
bills with short-term assets.
- A couple of other columns are stricter *flavors of* "current assets" --
some assets convert to cash faster than others, so these exclude inventory
(and sometimes receivables too) to focus on the most liquid resources.

A handful of attributes are given to you as ready-made ratios (their
exact formula is listed further below, near where the data is loaded).
Not every provided attribute is necessarily trustworthy -- where you can
cross-check a value against other information you have, it's worth doing.


# Import modules


In [1]:
## Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn

import os
import math

%matplotlib inline


# Get the data

- Each example is a row of data corresponding to a single company
- The column `Bankrupt` is 1 if the company subsequently went bankrupt; 0
if it did not go bankrupt
- The column `Id` is a Company Identifier
- The other columns (`num_01`, `num_02`, ..., `ratio_01`, `ratio_02`, ...,
`total_assets`, `sales`, `total_liabilities`, `short_term_liabilities`)
are financial attributes for the company; see the glossary above and the
attribute table further below
- `Id` is just a company identifier, not a feature -- don't feed it to
your model


In [2]:
# Data directory
DATA_DIR = "./Data"

data_file = "companies.csv"
data = pd.read_csv( os.path.join(DATA_DIR, "train", data_file) )

target_attr = "Bankrupt"
feature_cols = [c for c in data.columns if c not in (target_attr, "Id")]

n_samples, n_attrs = data.shape
print("Data shape: ", data.shape)

Data shape:  (4818, 68)


## Have a look at the data

Here's a peek.


In [3]:
data.head()


,ratio_03,ratio_16,ratio_22,ratio_15,ratio_21,ratio_05,ratio_19,ratio_02,ratio_18,ratio_11,...,num_29,num_01,num_16,num_35,num_06,num_23,num_25,num_21,num_12,num_32
0,-126.390,1.16500,4553.60,2.3941,1.09350,1.687400,0.096384,4.8078,191.430,0.79532,...,63.401805,4.100069,4.100069,13.943282,50.294332,50.293799,3.112625,82.632647,20.151757,33.154037
1,22.034,0.98183,3138.40,4.7595,1.15070,-0.068354,1.519500,4.8829,25.965,2.19900,...,101.455222,3.225688,3.725766,-1.991938,87.842433,87.846866,-3.646223,291.420069,1.603982,38.510487
2,35.766,1.05260,2400.00,2.2418,0.82243,0.334650,0.427330,4.5257,110.230,1.23330,...,36.442412,6.265857,5.457545,3.544788,50.064234,50.065799,3.626491,105.925154,3.091091,42.656700
3,438.200,1.02790,260.91,17.7410,0.94921,1.451000,1.370900,4.7352,50.118,0.54580,...,69.668787,8.980617,8.314934,6.783286,68.558246,68.558246,6.624087,46.750561,54.129560,60.569413
4,-71.398,1.29210,3455.90,2.0116,0.97245,1.062900,0.261370,4.0033,112.330,1.29210,...,27.547434,2.876196,2.875379,72.199004,28.966198,28.965085,0.055193,72.201630,0.391437,8.638643


In [4]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4818 entries, 0 to 4817
Data columns (total 68 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ratio_03                4808 non-null   float64
 1   ratio_16                4818 non-null   float64
 2   ratio_22                4812 non-null   float64
 3   ratio_15                4804 non-null   float64
 4   ratio_21                4744 non-null   float64
 5   ratio_05                4513 non-null   float64
 6   ratio_19                4735 non-null   float64
 7   ratio_02                4816 non-null   float64
 8   ratio_18                4776 non-null   float64
 9   ratio_11                4816 non-null   float64
 10  ratio_01                2750 non-null   float64
 11  ratio_25                4756 non-null   float64
 12  ratio_08                4818 non-null   float64
 13  ratio_10                4598 non-null   float64
 14  ratio_27                4787 non-null   

## Attributes given as raw figures

Each `num_##` column is a single financial figure for the company. Its
name below tells you which -- look it up in the glossary above for what
it means:

| Attribute | Financial Quantity |
|---|---|
| num_01 | Gross Profit Plus Depreciation |
| num_02 | EBITDA |
| num_03 | Gross Profit |
| num_04 | Operating Profit |
| num_05 | Gross Profit |
| num_06 | Current Assets |
| num_07 | Gross Profit Plus Interest |
| num_08 | Long-Term Capital Base |
| num_09 | Operating Profit |
| num_10 | Inventory |
| num_11 | Gross Profit (3-Year Cumulative) |
| num_12 | Near-Cash Assets |
| num_13 | Net Profit |
| num_14 | Owner's Equity |
| num_15 | Net Profit |
| num_16 | Net Profit Plus Depreciation |
| num_17 | Gross Profit (Adjusted) |
| num_18 | Profit on Sales |
| num_19 | Working Capital |
| num_20 | Retained Earnings |
| num_21 | Operating Expenses |
| num_22 | EBIT |
| num_23 | Current Assets |
| num_24 | EBITDA |
| num_25 | Gross Profit |
| num_26 | Owner's Equity (Beyond Initial Capital) |
| num_27 | Accounts Receivable |
| num_28 | Liabilities Net of Cash |
| num_29 | Owner's Equity |
| num_30 | Gross Profit Plus Interest |
| num_31 | Profit on Sales |
| num_32 | Quick Assets |
| num_33 | Gross Profit |
| num_34 | Gross Profit Plus Depreciation |
| num_35 | Operating Expenses |


## Attributes given as ratios

Some of the attributes are given to you as ready-made ratios. Their
formulas:

| Attribute | Formula |
|---|---|
| ratio_01 | (current assets - inventories) / long-term liabilities |
| ratio_02 | logarithm of total assets |
| ratio_03 | [(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365 |
| ratio_04 | long-term liabilities / equity |
| ratio_05 | profit on operating activities / financial expenses |
| ratio_06 | equity / fixed assets |
| ratio_07 | sales / fixed assets |
| ratio_08 | rotation receivables + inventory turnover in days |
| ratio_09 | total costs / total sales |
| ratio_10 | net profit / inventory |
| ratio_11 | total sales / total assets |
| ratio_12 | (short-term liabilities * 365) / cost of products sold |
| ratio_13 | (short-term liabilities * 365) / sales |
| ratio_14 | constant capital / fixed assets |
| ratio_15 | total assets / total liabilities |
| ratio_16 | sales / total assets |
| ratio_17 | sales / receivables |
| ratio_18 | (current liabilities * 365) / cost of products sold |
| ratio_19 | working capital / fixed assets |
| ratio_20 | sales / short-term liabilities |
| ratio_21 | sales (n) / sales (n-1) |
| ratio_22 | (total liabilities * 365) / (gross profit + depreciation) |
| ratio_23 | (current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation) |
| ratio_24 | sales / inventory |
| ratio_25 | total liabilities / ((profit on operating activities + depreciation) * (12/365)) |
| ratio_26 | working capital |
| ratio_27 | (inventory * 365) / cost of products sold |


# Evaluating your project

We will evaluate your submission on a test dataset that we provide
- It has no labels, so **you** can't use it to evaluate your model, but
**we** have the labels
- We will call this evaluation dataset the "holdout" data

Let's get it.


In [5]:
holdout_data = pd.read_csv( os.path.join(DATA_DIR, "holdout", data_file) )

print("Data shape: ", holdout_data.shape)


Data shape:  (1092, 67)


We will evaluate your model on the holdout examples using metrics
- Accuracy
- Recall
- Precision

You will likely have to make a trade-off between Recall and Precision.
Our evaluation of your submission will be partially based on how you made
(and described) that trade-off.

You may assume that missing an actual bankruptcy (a false negative) is 5
times worse than a false alarm (a false positive) -- i.e., failing to
flag a company that goes bankrupt costs 5x what incorrectly flagging a
healthy company costs.


# Establish a baseline

Before building your real model, establish a simple baseline: one that
ignores the given features entirely and always predicts the majority
class. This gives you a concrete lower bound -- any real model should
beat it by a meaningful margin, and comparing against it as you go (not
just at the end) is a standard, useful sanity check.

The train/test split and `cost` function below are reused for every
evaluation from here on, including your final model in the Submission
Summary -- so keep using these same `X_train`/`X_test`/`y_train`/`y_test`
(rather than creating a new split later) to keep every comparison you
make apples-to-apples.


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns=[target_attr]), data[target_attr], test_size=0.2,
    random_state=42, stratify=data[target_attr])

def cost(y_true, y_pred, fn_weight=5, fp_weight=1):
    """Default cost function matching the stated trade-off (5x worse to
    miss a bankruptcy). Adapt this if you use a different formulation --
    just apply whichever version you use consistently to both the
    baseline and your final model, so the ratio you report later stays
    meaningful."""
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn_weight * fn + fp_weight * fp

baseline_pipeline = DummyClassifier(strategy="most_frequent")
baseline_pipeline.fit(X_train, y_train)  # DummyClassifier ignores X entirely (incl. Id) --
                                          # your own final_pipeline below does need to ignore Id itself

baseline_cost = cost(y_test, baseline_pipeline.predict(X_test))
print(f"Baseline cost (always predicts the majority class): {baseline_cost}")


Baseline cost (always predicts the majority class): 310


# Your model

Time for you to continue on your own.


## Submission guidelines

Although your notebook may contain many models (e.g., due to your
iterative development) we will only evaluate a single model.

So choose one and obey the following guidelines. Add a markdown cell
right before `final_pipeline` is built explaining which model you chose
and why.


### What you submit: `final_pipeline`

Rather than asking you to hand-write a standalone prediction function, we
ask that your notebook define a single fitted object:

    final_pipeline

`final_pipeline` must support `.predict(X)`, where `X` is a raw pandas
DataFrame in exactly the same format as the data you were given -- same
columns (including `Id`), same dtypes -- including the holdout data
(`holdout_data`, loaded earlier -- we call it `X_hold` below to mirror
`X_train`/`X_test` naming, but it's the same object), which has no
`Bankrupt` column. If your approach can also produce probability
estimates (`.predict_proba(X)`), that's useful but not required -- any
classifier is acceptable, including ones that only produce hard class
predictions.

**We strongly recommend building `final_pipeline` as an
`sklearn.pipeline.Pipeline`** (optionally combined with
`ColumnTransformer`), with every step you need bundled in: any features
you engineer from the given columns, imputation, scaling, and finally your
classifier of choice. The final step doesn't have to be a standard
sklearn classifier -- a custom class is fine, as long as the object
`final_pipeline` as a whole exposes `.predict(X)` (and, optionally,
`.predict_proba(X)`).

The reason to do it this way, rather than writing your own function, is
that a fitted `Pipeline` *guarantees* the same transformations you apply
to your training data get applied identically to the holdout data when we
call `.predict()` -- there's no way for your train-time and
inference-time processing to accidentally drift apart, which is the main
risk in an exercise like this.

If part of your feature engineering doesn't fit neatly into a standard
sklearn transformer, you can wrap it in a
`sklearn.preprocessing.FunctionTransformer`, or write a small custom
transformer class with `fit`/`transform` methods -- both plug directly
into a `Pipeline`.

**Fit `final_pipeline` on `X_train`/`y_train`** (from the baseline
section above), not on all of `data` -- this keeps the Train/Test
comparison in the Submission Summary meaningful, since `X_test` stays
genuinely unseen by the exact object being graded.

We will evaluate your submission by calling

    y_hold_pred = final_pipeline.predict(X_hold)

to compute Accuracy, Recall, and Precision against the true holdout
labels, which only the instructors have. Since `.predict()` is what
determines your reported metrics, make sure its output reflects the
Recall/Precision trade-off you've decided on -- if `final_pipeline` also
exposes `.predict_proba(X)`, that trade-off might be baked in as a chosen
threshold; if not, it needs to be reflected however your method decides
between classes.

Before considering your notebook done, actually run
`final_pipeline.predict(holdout_data)` yourself (further down, in the
Submission Summary) and confirm it returns a prediction for every row
without error -- catching a shape/dtype mismatch yourself is much better
than discovering one at grading time.


### Notebook style

Your notebook will probably have lots of code that does not contribute to
building `final_pipeline`.

We don't want to have to run the entire notebook or figure out which cells
are prerequisites. As a matter of good coding practice, please organize
your notebook (and use markdown) so that a single, clearly-marked run
through the cells builds `final_pipeline` from scratch, without requiring
us to untangle your exploratory work. You don't need to delete your
exploratory cells -- labeling them clearly (e.g. under an "## Exploration"
heading, separate from the cells that actually build `final_pipeline`) is
enough.


# Submission summary (required)

This section is **required**, not optional -- we grade directly from what
it prints, so make sure it's reproducible by running the notebook top to
bottom. It has several parts: a performance table (train vs. your
held-out test split, reusing `X_train`/`X_test`/`y_train`/`y_test` from
the baseline section above), a baseline comparison (which also prints
the size of your test split, its confusion matrix, and the cost of a
missed bankruptcy and of a false alarm under your `cost` function), a
feature-correlation check, a final sanity check against the real holdout
file, and a short written report. Fill in every blank.


In [7]:
import warnings
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (HistGradientBoostingClassifier, RandomForestClassifier,
                              VotingClassifier)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [8]:
RANDOM_STATE = 42
FN_COST, FP_COST = 5, 1                                        # 5X for False Negative, 1X for False Positive
BAYES_THRESHOLD = FP_COST / (FP_COST + FN_COST)                # = 1/6, optimal for perfectly calibrated probabilities
THRESHOLD_GRID = np.round(np.arange(0.01, 0.951, 0.005), 3)    #List of cut-offs tested, from 0.01 to 0.95 in steps of 0.005.

# Make sure the `cost` function from the baseline section uses the same weights.
assert cost(np.array([1]), np.array([0])) == FN_COST and cost(np.array([0]), np.array([1])) == FP_COST

NUM_COLS = [f"num_{i:02d}" for i in range(1, 36)]
RATIO_COLS = [f"ratio_{i:02d}" for i in range(1, 28)]
BASE_COLS = ["total_assets", "sales", "total_liabilities", "short_term_liabilities"]
RAW_INPUT_COLS = NUM_COLS + RATIO_COLS + BASE_COLS

# Record Financial Quantity of same name but different Attribute
# First of Column = "primary" one used for named ratios. 
# The others stay available as separate features (order left to right).
QUANTITY_COLUMNS = {
    "gross_profit_plus_dep": ["num_01", "num_34"],
    "ebitda": ["num_02", "num_24"],
    "gross_profit": ["num_03", "num_05", "num_25", "num_33"],
    "operating_profit": ["num_04", "num_09"],
    "current_assets": ["num_06", "num_23"],
    "gross_profit_plus_int": ["num_07", "num_30"],
    "long_term_capital": ["num_08"],
    "inventory": ["num_10"],
    "gross_profit_3y": ["num_11"],
    "near_cash_assets": ["num_12"],
    "net_profit": ["num_13", "num_15"],
    "equity": ["num_14", "num_29"],
    "net_profit_plus_dep": ["num_16"],
    "gross_profit_adj": ["num_17"],
    "profit_on_sales": ["num_18", "num_31"],
    "working_capital": ["num_19"],
    "retained_earnings": ["num_20"],
    "operating_expenses": ["num_21", "num_35"],
    "ebit": ["num_22"],
    "equity_beyond_capital": ["num_26"],
    "receivables": ["num_27"],
    "liabilities_net_of_cash": ["num_28"],
    "quick_assets": ["num_32"],
}

#Checking all 35 num_ are covered
assert sorted(c for cols in QUANTITY_COLUMNS.values() for c in cols) == NUM_COLS

In [9]:
# ---- BUILD step 1: feature engineering transformer --------------------------------------

# Ensure all value are Float
# Then: Invalid, missing, or infinite values --> NaN
def _num(s):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    return s.where(np.isfinite(s))

# Safe division: When denominator is 0 or NaN; result is not finite --> NaN.
def _div(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        out = a / b.where(b != 0)
    return out.where(np.isfinite(out))

# Signed log1p: keeps sign, compresses heavy tails, ~identity near 0
def _slog(x):
    return np.sign(x) * np.log1p(np.abs(x))

# Of QUANTITY_COLUMNS: Compare if two tables match within 2% (value of the same row are within 2% difference)
# If both _nums have less than 30 overlap --> return NaN = No evidence they agree in value
def agreement_rate(a, b, rtol=0.02, atol=1e-9, min_overlap=30):
    m = a.notna() & b.notna()
    if m.sum() < min_overlap:
        return np.nan
    a, b = a[m].to_numpy(float), b[m].to_numpy(float)
    return float((np.abs(a - b) <= rtol * np.maximum(np.abs(a), np.abs(b)) + atol).mean())

# Check data is Pandas DataFrame
# Organize columns, ensure (besides header) are all values, start at 0,1,2...
def prepare_raw(X):
    if not isinstance(X, pd.DataFrame):
        raise TypeError("expected a pandas DataFrame with the raw columns")
    return X.reindex(columns=RAW_INPUT_COLS).apply(_num).reset_index(drop=True)

In [10]:
class FinancialFeatures(TransformerMixin, BaseEstimator):
    
    """Raw company table -> candidate features (original columns + safe, scale-free ratios),
    then constant / mostly-missing / highly-correlated candidates are pruned at fit time.

    Unsupervised: fit() never looks at y. It learns only 
        (a) which features survive constant / almost-all-missing / collinearity pruning 
        (b) if consensus=True, which documented duplicate groups may be merged.

    consensus=False (default): duplicate sources are never merged; named ratios use the
    first-listed column of each quantity, and every column remains as its own feature.
    
    consensus=True (optional experiment): a duplicate group is merged (row-wise median)
    only if EVERY pair in the group has enough overlapping rows AND agrees on at least
    `agree_min` of them. Missing evidence or any disagreeing pair -> not merged.
    """

    def __init__(self, consensus=False, agree_rtol=0.02, agree_min=0.95, min_overlap=30,
                 corr_threshold=0.95, max_missing_rate=0.98):
        self.consensus = consensus
        self.agree_rtol = agree_rtol
        self.agree_min = agree_min
        self.min_overlap = min_overlap
        self.corr_threshold = corr_threshold
        self.max_missing_rate = max_missing_rate

    # Merge two multiple _nums of same financial quanitity name together if they agree. 
    # Don't merge if they don't agree, consider as two different features
    def _mergeable_groups(self, raw):
        merged = {}
        for name, cols in QUANTITY_COLUMNS.items():
            if len(cols) < 2:
                continue
            rates = [agreement_rate(raw[a], raw[b], self.agree_rtol, min_overlap=self.min_overlap)
                     for i, a in enumerate(cols) for b in cols[i + 1:]]
            if all(np.isfinite(r) and r >= self.agree_min for r in rates):   # complete agreement only
                merged[name] = cols
        return merged

    def _quantity(self, raw, name):
        cols = QUANTITY_COLUMNS[name]
        if name in self.merged_groups_:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)   # all-NaN row -> NaN
                return pd.Series(np.nanmedian(raw[cols].to_numpy(), axis=1))
        return raw[cols[0]]

    def _engineer(self, raw):
        q = {k: self._quantity(raw, k) for k in QUANTITY_COLUMNS}
        TA, TL, STL, S = (raw["total_assets"], raw["total_liabilities"],
                          raw["short_term_liabilities"], raw["sales"])
        LTL = TL - STL
        f = {}
        
        # (A) Altman Z' (private-firm weights) = estimates the Probability of Bankruptcy
        # These ratios cover the main drivers of bankruptcy
        f["liabilities_to_assets"] = _div(TL, TA)
        f["working_capital_to_assets"] = _div(q["working_capital"], TA)
        f["retained_earnings_to_assets"] = _div(q["retained_earnings"], TA)
        f["ebit_to_assets"] = _div(q["ebit"], TA)
        f["equity_to_liabilities"] = _div(q["equity"], TL)
        f["sales_to_assets"] = _div(S, TA)
        
        # >2.9 = low probability of bankruptcy
        # 1.23 - 2.9 = Moderate probablity of bankruptcy
        # <1.23 = High probability of bankruptcy
        f["altman_z_prime_approx"] = (0.717 * f["working_capital_to_assets"]
                                      + 0.847 * f["retained_earnings_to_assets"]
                                      + 3.107 * f["ebit_to_assets"]
                                      + 0.420 * f["equity_to_liabilities"]
                                      + 0.998 * f["sales_to_assets"])

        # Other Ratios
        f["net_profit_to_assets"] = _div(q["net_profit"], TA)
        f["current_ratio"] = _div(q["current_assets"], STL)
        f["quick_ratio"] = _div(q["quick_assets"], STL)
        f["near_cash_to_stl"] = _div(q["near_cash_assets"], STL)
        f["working_capital_to_stl"] = _div(q["working_capital"], STL)
        f["stl_to_assets"] = _div(STL, TA)
        f["ltl_to_assets"] = _div(LTL, TA)
        f["stl_share_of_liabilities"] = _div(STL, TL)
        f["equity_to_assets"] = _div(q["equity"], TA)
        f["liabilities_net_of_cash_to_assets"] = _div(q["liabilities_net_of_cash"], TA)
        f["net_profit_dep_to_liabilities"] = _div(q["net_profit_plus_dep"], TL)
        f["ebitda_to_liabilities"] = _div(q["ebitda"], TL)
        f["gross_profit_dep_to_liabilities"] = _div(q["gross_profit_plus_dep"], TL)
        f["ebit_to_stl"] = _div(q["ebit"], STL)
        f["net_margin"] = _div(q["net_profit"], S)
        f["gross_margin"] = _div(q["gross_profit"], S)
        f["operating_margin"] = _div(q["operating_profit"], S)
        f["ebit_margin"] = _div(q["ebit"], S)
        f["opex_to_sales"] = _div(q["operating_expenses"], S)
        f["receivable_days_sales"] = _div(q["receivables"] * 365, S)
        f["inventory_days_sales"] = _div(q["inventory"] * 365, S)
        f["stl_days_sales"] = _div(STL * 365, S)
        f["sales_to_stl"] = _div(S, STL)
        
        # (B) every original raw figure relative to size (each duplicate kept separately)
        # Removes the effect of company sizes
        for c in NUM_COLS:
            f[f"{c}_to_assets"] = _div(raw[c], TA)
       
        # (C) every provided ratio, as given
        for c in RATIO_COLS:
            f[c] = raw[c]
        
        # (D) every original raw column (sales/liabilities etc. carry size information)
        for c in NUM_COLS + BASE_COLS:
            f[c] = raw[c]
        feats = pd.DataFrame(f).apply(_slog)

            
        # Not slog-transformed (already on a sensible scale):
        # Flagging when see sign of bankruptcy: 
        """negative equity, Shrinking Profit, Negative Working Capital = Cushion, Total Lability > Total Asset """
        feats.insert(0, "log_total_assets", np.log(TA.where(TA > 0)))
        known = lambda *s: np.logical_and.reduce([x.notna() for x in s])
        feats["flag_negative_equity"] = (q["equity"] < 0).astype(float).where(known(q["equity"]))
        feats["flag_net_loss"] = (q["net_profit"] < 0).astype(float).where(known(q["net_profit"]))
        feats["flag_negative_working_capital"] = ((q["working_capital"] < 0).astype(float)
                                                  .where(known(q["working_capital"])))
        feats["flag_liabilities_exceed_assets"] = (TL > TA).astype(float).where(known(TL, TA))
        feats["n_missing_raw"] = raw.isna().sum(axis=1).astype(float)
        return feats

    # After merges, drop columns of 98% agreeing data (of name Financial Quanitity)
    # Leave only one out multiple Feature if they have high corrlation, r >|0.95|  --> record it 
    def fit(self, X, y=None):                      # y is accepted for Pipeline API, never used
        raw = prepare_raw(X)
        self.merged_groups_ = self._mergeable_groups(raw) if self.consensus else {}
        feats = self._engineer(raw)
        keep = [c for c in feats.columns
                if feats[c].nunique(dropna=True) > 1 and feats[c].isna().mean() <= self.max_missing_rate]
        corr = feats[keep].corr().abs()
        selected = []
        for col in keep:                           # greedy, in the fixed priority order above
            if not selected or not (corr.loc[col, selected] > self.corr_threshold).any():
                selected.append(col)
        self.dropped_collinear_ = [c for c in keep if c not in selected]
        self.feature_names_out_ = np.array(selected, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        out = self._engineer(prepare_raw(X))[list(self.feature_names_out_)]
        out.index = X.index
        return out

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

In [11]:
# ---- BUILD step 2: decision-threshold wrapper + candidate models -----------------------


class ThresholdClassifier(ClassifierMixin, BaseEstimator):
    """Wraps a probabilistic classifier and predicts class 1 when P(bankrupt) >= threshold.
    predict_proba() is passed through unchanged; predict() carries the cost trade-off."""

    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator).fit(X, y)
        self.classes_ = self.estimator_.classes_
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)

    def predict(self, X):
        return np.where(self.predict_proba(X)[:, 1] >= self.threshold, self.classes_[1], self.classes_[0])


def make_classifier(name):    
    # Gradient boosting (shallow and deeper):
    hgb_shallow = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=15, min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE)

    hgb_deep = HistGradientBoostingClassifier(
        learning_rate=0.03, max_iter=600, max_leaf_nodes=31, min_samples_leaf=20,
        l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE)

    #Random Forest
    rf = Pipeline([("impute", SimpleImputer(strategy="median")),
                   ("rf", RandomForestClassifier(n_estimators=500, min_samples_leaf=10, max_features="sqrt",
                                                 n_jobs=-1, random_state=RANDOM_STATE))])
    # Logistic Regresson
    lr = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                   ("scale", StandardScaler()),
                   ("lr", LogisticRegression(C=0.1, max_iter=5000))])
    return {
        "LogisticRegression": lr,
        "RandomForest": rf,
        "HistGradientBoosting (shallow)": hgb_shallow,
        "HistGradientBoosting (deeper)": hgb_deep,
        "Soft-vote ensemble (HGB + RF + LR)": VotingClassifier([("hgb", hgb_shallow), ("rf", rf), ("lr", lr)], voting="soft"),
    }[name]

CANDIDATES = ["LogisticRegression", "RandomForest", "HistGradientBoosting (shallow)",
              "HistGradientBoosting (deeper)", "Soft-vote ensemble (HGB + RF + LR)"]

def make_pipeline(name, threshold=None, consensus=False):
    clf = make_classifier(name)
    if threshold is not None:
        clf = ThresholdClassifier(clf, threshold=threshold)
    return Pipeline([("features", FinancialFeatures(consensus=consensus)), ("model", clf)])

# Out-of-fold P(bankrupt). Every fitted step is re-fitted inside each training fold.
def oof_proba(name, X, y, seed, consensus=False):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    return cross_val_predict(make_pipeline(name, consensus=consensus), X, y, cv=cv,
                             method="predict_proba")[:, 1]

# 0 for healthy 
# 1 for Bankruptcy 
def cost_at(y, p, t):
    return cost(np.asarray(y), (p >= t).astype(int))

# Go through the list of companies to see if they are bankrupt or not
def cost_curve(y, p):
    return np.array([cost_at(y, p, t) for t in THRESHOLD_GRID], dtype=float)

# Smooth curve by moving average
def smooth(v, window=7):
    return pd.Series(v).rolling(window, center=True, min_periods=1).mean().to_numpy()

In [15]:
# ---- BUILD step 3: compare candidates with out-of-fold predictions on X_train only -----
y_tr = y_train.to_numpy()
oof = {}
rows = []
for name in CANDIDATES:
    p = oof_proba(name, X_train, y_tr, seed=RANDOM_STATE)
    oof[name] = p
    raw_curve = cost_curve(y_tr, p)
    t_sel = THRESHOLD_GRID[smooth(raw_curve).argmin()]
    rows.append(dict(model=name,
                     roc_auc=roc_auc_score(y_tr, p),
                     pr_auc=average_precision_score(y_tr, p),
                     cost_at_bayes_t=cost_at(y_tr, p, BAYES_THRESHOLD),
                     selected_t=t_sel,
                     actual_cost_at_selected_t=raw_curve[THRESHOLD_GRID == t_sel][0]))
cv_results = (pd.DataFrame(rows).set_index("model")
              .sort_values(["actual_cost_at_selected_t", "pr_auc"], ascending=[True, False]))
oof_baseline_cost = FN_COST * int(y_tr.sum())          # always predicting "not bankrupt"
print(f"5-fold OOF results on X_train ({len(y_tr)} rows, {int(y_tr.sum())} bankrupt; "
      f"majority-class cost on the same rows = {oof_baseline_cost}).")

print("selected_t = argmin of the smoothed cost curve; the cost column is the ACTUAL cost at that t.")

print(cv_results.round(4).to_string())

best_name = cv_results.index[0]
print(f"\nSelected model: {best_name}")

5-fold OOF results on X_train (3854 rows, 246 bankrupt; majority-class cost on the same rows = 1230).
selected_t = argmin of the smoothed cost curve; the cost column is the ACTUAL cost at that t.
                                    roc_auc  pr_auc  cost_at_bayes_t  selected_t  actual_cost_at_selected_t
model                                                                                                      
HistGradientBoosting (shallow)       0.9602  0.7880              453       0.050                      402.0
HistGradientBoosting (deeper)        0.9566  0.7828              463       0.030                      419.0
Soft-vote ensemble (HGB + RF + LR)   0.9432  0.7355              460       0.195                      433.0
LogisticRegression                   0.9129  0.6361              611       0.205                      597.0
RandomForest                         0.8912  0.4295              711       0.170                      724.0

Selected model: HistGradientBoosting (shallow)


In [16]:
# ---- BUILD step 4: choose the decision threshold for the 5:1 cost --------------------
# Three different 5-fold splits (the first is reused from step 3). 
# The threshold is the argmin of the run-averaged cost curve after smoothing over neighbouring thresholds.
seeds = [RANDOM_STATE, RANDOM_STATE + 1, RANDOM_STATE + 2]
oof_runs = [oof[best_name]] + [oof_proba(best_name, X_train, y_tr, seed=s) for s in seeds[1:]]
mean_curve = np.mean([cost_curve(y_tr, p) for p in oof_runs], axis=0)     # actual costs
smoothed_curve = smooth(mean_curve)
chosen_threshold = float(THRESHOLD_GRID[smoothed_curve.argmin()])

actual_costs = [int(cost_at(y_tr, p, chosen_threshold)) for p in oof_runs]
costs_at_half = [int(cost_at(y_tr, p, 0.5)) for p in oof_runs]

# One confusion matrix from probabilities AVERAGED over the 3 runs (a different calculation:
# its cost need not equal the mean of the per-run costs).
oof_mean = np.mean(oof_runs, axis=0)
oof_pred = (oof_mean >= chosen_threshold).astype(int)
tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_tr, oof_pred).ravel()

# Error analysis: what do the missed bankruptcies look like?
miss_raw = prepare_raw(X_train).isna().sum(axis=1).to_numpy()
fn_mask, tp_mask = (y_tr == 1) & (oof_pred == 0), (y_tr == 1) & (oof_pred == 1)
err_note = (f"missed bankruptcies (FN) have mean P(bankrupt)={oof_mean[fn_mask].mean():.3f} and "
            f"{miss_raw[fn_mask].mean():.1f} missing raw fields, vs caught ones (TP) "
            f"P={oof_mean[tp_mask].mean():.3f} and {miss_raw[tp_mask].mean():.1f} missing fields"
            if fn_mask.any() and tp_mask.any() else "no FN/TP split to analyse")

# ---- Display ----
TABLE_STYLE = [
    {"selector": "caption", "props": "caption-side: top; text-align: left; font-weight: bold; padding: 6px 0;"},
    {"selector": "th, td", "props": "padding: 4px 12px; border-bottom: 1px solid #ddd;"},
    {"selector": "td", "props": "text-align: right;"},
    {"selector": "th.row_heading", "props": "text-align: left;"},
]
print(f"Chosen threshold: {chosen_threshold:.3f}   (theoretical Bayes threshold 1/6 = {BAYES_THRESHOLD:.3f})")
print(f"Selected from the smoothed mean cost curve (smoothed value {smoothed_curve.min():.1f}); "
      f"costs below are actual.\n")

runs = [f"Run {i + 1}" for i in range(len(oof_runs))]
cost_table = pd.DataFrame(
    [actual_costs + [np.mean(actual_costs)],
     costs_at_half + [np.mean(costs_at_half)],
     [oof_baseline_cost] * len(runs) + [oof_baseline_cost]],
    index=[f"Chosen threshold ({chosen_threshold:.3f})", "Default threshold (0.5)",
           "Majority-class baseline"],
    columns=runs + ["Mean"])
display(cost_table.style.format("{:.0f}", subset=runs).format("{:.1f}", subset=["Mean"])
        .set_caption("Out-of-fold cost (5·FN + 1·FP) on X_train").set_table_styles(TABLE_STYLE))

confusion_table = pd.DataFrame([[tn_o, fp_o], [fn_o, tp_o]],
                               index=["Actual: healthy", "Actual: bankrupt"],
                               columns=["Predicted: healthy", "Predicted: bankrupt"])
display(confusion_table.style.set_caption(
    f"Confusion matrix from run-averaged probabilities: cost {FN_COST*fn_o + FP_COST*fp_o}, "
    f"recall {tp_o/max(tp_o+fn_o,1):.1%}, precision {tp_o/max(tp_o+fp_o,1):.1%}")
        .set_table_styles(TABLE_STYLE))

if fn_mask.any() and tp_mask.any():
    error_table = pd.DataFrame(
        {"Companies": [int(fn_mask.sum()), int(tp_mask.sum())],
         "Mean P(bankrupt)": [oof_mean[fn_mask].mean(), oof_mean[tp_mask].mean()],
         "Mean missing raw fields": [miss_raw[fn_mask].mean(), miss_raw[tp_mask].mean()]},
        index=["Missed bankruptcies (FN)", "Caught bankruptcies (TP)"])
    display(error_table.style.format({"Mean P(bankrupt)": "{:.3f}", "Mean missing raw fields": "{:.1f}"})
            .set_caption("Error analysis: missed vs caught bankruptcies").set_table_styles(TABLE_STYLE))

Chosen threshold: 0.055   (theoretical Bayes threshold 1/6 = 0.167)
Selected from the smoothed mean cost curve (smoothed value 406.0); costs below are actual.



,Run 1,Run 2,Run 3,Mean
Chosen threshold (0.055),408,408,406,407.3
Default threshold (0.5),577,547,575,566.3
Majority-class baseline,1230,1230,1230,1230.0


,Predicted: healthy,Predicted: bankrupt
Actual: healthy,3477,131
Actual: bankrupt,47,199


,Companies,Mean P(bankrupt),Mean missing raw fields
Missed bankruptcies (FN),47,0.013,1.1
Caught bankruptcies (TP),199,0.675,1.8


In [17]:
# ---- BUILD step 5: fit final_pipeline on X_train/y_train ------------------------------
final_pipeline = make_pipeline(best_name, threshold=chosen_threshold)
final_pipeline.fit(X_train, y_train)

def describe_model(est):
    """Concrete complexity description of a fitted classifier."""
    if isinstance(est, Pipeline):
        return describe_model(est.steps[-1][1])
    if isinstance(est, HistGradientBoostingClassifier):
        try:   # realised tree sizes (private attribute; skipped if sklearn changes it)
            n_leaves = [p[0].get_n_leaf_nodes() for p in est._predictors]
            depths = [p[0].get_max_depth() for p in est._predictors]
            realised = f" (mean {np.mean(n_leaves):.1f} leaves, max depth {max(depths)})"
        except Exception:
            realised = ""
        return (f"HistGradientBoosting: {est.n_iter_} boosting iterations (trees), max_leaf_nodes="
                f"{est.max_leaf_nodes}{realised}, min_samples_leaf={est.min_samples_leaf}, "
                f"learning_rate={est.learning_rate}, l2={est.l2_regularization}")
    if isinstance(est, RandomForestClassifier):
        depths = [t.get_depth() for t in est.estimators_]
        return (f"RandomForest: {len(est.estimators_)} trees\n"
                f"max_depth: None\n"
                f"realised depth: {min(depths)}-{max(depths)} (mean"
                f" {np.mean(depths):.1f})\n"
                f"min_samples_leaf: {est.min_samples_leaf}\n"
                f"max_features: {est.max_features}")

    if isinstance(est, LogisticRegression):
        return (f"LogisticRegression: L2 penalty, C={est.C}, {int(np.sum(est.coef_ != 0))} non-zero "
                f"coefficients of {est.coef_.size}")
    
    if isinstance(est, VotingClassifier):
        return "Soft-vote average of [" + "; ".join(describe_model(e) for e in est.estimators_) + "]"
    return type(est).__name__

final_features = final_pipeline.named_steps["features"]
final_model = final_pipeline.named_steps["model"]
model_size_desc = describe_model(final_model.estimator_)
print("final_pipeline fitted.")
print(f"  features used: {len(final_features.feature_names_out_)} "
      f"(dropped {len(final_features.dropped_collinear_)} as |r| > {final_features.corr_threshold})")
print(f"  decision threshold: {final_model.threshold:.3f}")
print(f"  model: {model_size_desc}")

final_pipeline fitted.
  features used: 97 (dropped 40 as |r| > 0.95)
  decision threshold: 0.055
  model: HistGradientBoosting: 300 boosting iterations (trees), max_leaf_nodes=15 (mean 15.0 leaves, max depth 12), min_samples_leaf=40, learning_rate=0.05, l2=1.0


In [18]:
# Uses X_train/X_test/y_train/y_test from the "Establish a baseline"
# section above. X_train/X_test include `Id` -- the same format
# final_pipeline.predict() will actually be called with -- your pipeline
# should ignore Id internally, the same way it will need to for the real
# holdout call.
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                              roc_auc_score, average_precision_score)

y_train_pred = final_pipeline.predict(X_train)
y_test_pred = final_pipeline.predict(X_test)

print(f"{'Split':8s} {'Accuracy':>10s} {'Recall':>10s} {'Precision':>10s}")
for split_name, y_true, y_pred in [("Train", y_train, y_train_pred), ("Test", y_test, y_test_pred)]:
    print(f"{split_name:8s} {accuracy_score(y_true, y_pred):>10.1%} "
          f"{recall_score(y_true, y_pred):>10.1%} {precision_score(y_true, y_pred):>10.1%}")

# ROC-AUC / PR-AUC (train vs test) if your final_pipeline supports
# predict_proba -- these are the clearest signal of overfitting (compare
# the train row to the test row) and of ranking quality under class
# imbalance. If your model doesn't support predict_proba, this is skipped
# automatically -- explain why in the report below instead.
if hasattr(final_pipeline, "predict_proba"):
    proba_train = final_pipeline.predict_proba(X_train)[:, 1]
    proba_test = final_pipeline.predict_proba(X_test)[:, 1]
    print(f"\n{'Split':8s} {'ROC-AUC':>10s} {'PR-AUC':>10s}")
    print(f"{'Train':8s} {roc_auc_score(y_train, proba_train):>10.4f} "
          f"{average_precision_score(y_train, proba_train):>10.4f}")
    print(f"{'Test':8s} {roc_auc_score(y_test, proba_test):>10.4f} "
          f"{average_precision_score(y_test, proba_test):>10.4f}")
else:
    print("\nfinal_pipeline has no predict_proba -- ROC-AUC/PR-AUC not reported "
          "(explain why in the report below).")


Split      Accuracy     Recall  Precision
Train         99.4%     100.0%      91.1%
Test          95.2%      87.1%      58.7%

Split       ROC-AUC     PR-AUC
Train        1.0000     1.0000
Test         0.9815     0.8788


In [19]:
# Baseline comparison, using the same `cost` function and test split as
# the baseline established earlier.
from sklearn.metrics import confusion_matrix

final_cost = cost(y_test, y_test_pred)
improvement_ratio = baseline_cost / final_cost if final_cost > 0 else float("inf")

print(f"Baseline cost:        {baseline_cost}")
print(f"Final model cost:     {final_cost}")
print(f"Improvement ratio (baseline / final): {improvement_ratio:.2f}")

# Context for the numbers above, so they can be checked.
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
print(f"\nTest split: {len(y_test)} rows, {int(y_test.sum())} bankrupt")
print(f"Confusion matrix (test): TN={tn} FP={fp} FN={fn} TP={tp}")
# The weights your `cost` function applies, read from the function itself:
print(f"Cost of one missed bankruptcy (FN): {cost(np.array([1]), np.array([0]))}")
print(f"Cost of one false alarm (FP):       {cost(np.array([0]), np.array([1]))}")


Baseline cost:        310
Final model cost:     78
Improvement ratio (baseline / final): 3.97

Test split: 964 rows, 62 bankrupt
Confusion matrix (test): TN=864 FP=38 FN=8 TP=54
Cost of one missed bankruptcy (FN): 5
Cost of one false alarm (FP):       1


In [20]:
# Highest pairwise correlation among your final feature set. Set
# X_features to whatever DataFrame of columns you actually feed your
# model (post feature-engineering, pre-imputation/scaling is fine).

X_features = X_train[feature_cols]  # YOUR CODE HERE: replace if your features differ from the raw columns

corr = X_features.corr(numeric_only=True).abs().to_numpy(copy=True)
np.fill_diagonal(corr, 0)
corr = pd.DataFrame(corr, index=X_features.columns, columns=X_features.columns)
max_corr = corr.to_numpy().max()
worst_pair = corr.stack().idxmax()
print(f"Highest pairwise correlation among final features: {max_corr:.4f} ({worst_pair[0]} vs {worst_pair[1]})")


Highest pairwise correlation among final features: 1.0000 (num_22 vs num_25)


In [21]:
# Final sanity check: run final_pipeline against the actual holdout file,
# exactly as it will be called for grading. This should complete with no
# error and return one prediction per row -- if it doesn't, fix that
# before considering the notebook done.

X_hold = holdout_data
y_hold_pred = final_pipeline.predict(X_hold)
assert len(y_hold_pred) == len(X_hold), "predict() didn't return one row per holdout example"
print(f"OK -- predicted {len(y_hold_pred)} rows, bankrupt rate {y_hold_pred.mean():.1%}")


OK -- predicted 1092 rows, bankrupt rate 11.8%


In [22]:
# Written report -- fill in every field. Each is read directly for
# grading, so answer specifically enough that someone re-reading your
# notebook (without re-running your search/tuning process) can verify it.

#Writing the result for comparison:
_cv_line = "; ".join(f"{m}: PR-AUC {r.pr_auc:.3f}, OOF cost {r.actual_cost_at_selected_t:.0f} at t={r.selected_t:.3f}"
                     for m, r in cv_results.iterrows())
_auc_tr = roc_auc_score(y_train, final_pipeline.predict_proba(X_train)[:, 1])
_auc_te = roc_auc_score(y_test, final_pipeline.predict_proba(X_test)[:, 1])

submission_summary = {
    "approach_summary":
    f"(1) Established a majority-class baseline: test cost {baseline_cost}.\n"
    f"(2) Data handling: Converted invalid and infinite values to missing values. "
    f"Kept potentially conflicting financial columns separate rather than automatically merging them.\n"
    f"(3) Feature engineering: Added safe, scale-free ratios "
    f"(leverage, liquidity, profitability, coverage, and turnover), "
    f"an approximate Altman Z' score, distress flags, and a missing-field count. "
    f"Removed constant, mostly missing, and highly correlated features using training data only.\n"
    f"(4) Compared {len(cv_results)} pipelines using 5-fold stratified "
    f"cross-validation on X_train, with every fitted preprocessing step "
    f"re-fitted inside each training fold. Results: {_cv_line}. "
    f"Selected {best_name} based on actual OOF cost at each candidate's "
    f"selected threshold, with average precision as the tie-breaker.\n"
    f"(5) Selected the final decision threshold by minimising a smoothed "
    f"mean OOF cost curve across 3 CV splits. Reported actual, unsmoothed costs.\n"
    f"(6) Examined errors using run-averaged OOF predictions: {err_note}.",
    
    # Briefly summarize your modeling process: what did you try, in what
    # order, and what did you learn or change at each step? (e.g. which
    # models/approaches you compared, what looking at your errors showed
    # you, and how that shaped what you tried next.) A few sentences or a
    # short list is fine -- however extensive your actual process was.

    "model_type": 
    f"Tested Model Candidates: Logistic Regression, Random Forest, HistGradientBoosting (deep and shallow), Soft-vote ensemble (average).\n"
    f"Final Model: {best_name} wrapped in a cost-tuned ThresholdClassifier inside an sklearn Pipeline",
    # e.g. "RandomForestClassifier" or "LogisticRegression"

    "model_size": 
    f"FINAL MODEL: {best_name}\n"
    f"{model_size_desc}\n"
    f"Input: {len(final_features.feature_names_out_)}\n "
    f"features after unsupervised pruning.\n"
    f"Train ROC-AUC {_auc_tr:.3f} vs test ROC-AUC {_auc_te:.3f}.\n"
    f"Commnet: the model fits the training data almost perfectly, so there is some overfitting,\n"
    f"The gap is largest for PR-AUC and precision.\n"
    f"Stronger regularisation (fewer trees or larger leaves) could narrow the gap, but would need to be checked with cross-validation.",
    

    # State the complexity of your FINAL model concretely: for a tree
    # ensemble, number of trees and max_depth; for a linear model,
    # penalty type, regularization strength, and number of nonzero
    # coefficients; for PCA or another dimensionality-reduction step (if
    # used), number of components retained. Use whatever is the
    # analogous complexity measure for your model type. Comment briefly
    # on how this compares to your train-vs-test numbers above.

    "feature_engineering_summary": 
    f"Base Features & Duplicates: Included original financial columns, all provided ratios, \n"
    f"and raw figures divided by total assets (kept duplicates separate without automatic merging).\n"
    f"Engineered Ratios & Metrics: Liabilities/assets, working capital/assets, and EBIT/assets,\n"
    f"Current and quick ratios, and near-cash/short-term liabilities\n"
    f"Profit margins and receivable/inventory days on sales\n"
    f"An approximate Altman Z' score\n"
    f"Additional Indicators: Added financial distress flags and a missing-field count.\n"
    f"Transformation/Data Cleaning: Guarded all division operations against $0$, NaN, and infintiy values (mapping them to NaN.\n"
    f"Applied a signed log1p transformation to compress heavy tails, with a safeguard to ensure log_total_assets is not transformed twice.\n"
    f"Feature Pruning: remove constant, mostly missing, high correlation (|r|>0.95).",
    # 2-4 sentences: what transformations or new features (if any) did
    # you create from the given attributes, and why (or why not)?

    "imbalance_handling": 
    f"Training bankruptcy rate is {y_train.mean():.1%}. I did not use resampling or class weights,\n"
    f"because both would distort the probabilities. Instead: stratified splits and CV, model selection\n"
    f"on expected cost and PR-AUC rather than accuracy, and a cost-tuned decision threshold\n"
    f"(selected: {chosen_threshold:.3f}, vs the default 0.5).",
    # How, if at all, did you address the class imbalance in the
    # training data?

    "cost_ratio_handling": 
    f"final_pipeline.predict() flags a company when P(bankrupt) >= {chosen_threshold:.3f}. "
    f"Theory gives 1/(1+5) = {BAYES_THRESHOLD:.3f} for calibrated probabilities. I picked the threshold\n"
    f"empirically by minimising a smoothed mean OOF cost curve (5*FN + 1*FP on out-of-fold\n"
    f"predictions on X_train, averaged over 3 5-fold splits, then a moving average over neighbouring\n "
    f"thresholds so the choice does not land on a lucky spike). I report the actual, unsmoothed mean\n "
    f"OOF cost at that threshold: {np.mean(actual_costs):.1f}, vs\n"
    f"{np.mean(costs_at_half):.1f} at 0.5 and {oof_baseline_cost} for the majority baseline.\n"
    f"X_test was never used to choose it.",
    # Explain specifically how final_pipeline.predict()'s output
    # reflects the stated cost asymmetry between false negatives and
    # false positives (e.g., a chosen decision threshold and how you
    # picked it, or another mechanism).
}

for key, value in submission_summary.items():
    print(f"{key}:\n  {value}\n")


approach_summary:
  (1) Established a majority-class baseline: test cost 310.
(2) Data handling: Converted invalid and infinite values to missing values. Kept potentially conflicting financial columns separate rather than automatically merging them.
(3) Feature engineering: Added safe, scale-free ratios (leverage, liquidity, profitability, coverage, and turnover), an approximate Altman Z' score, distress flags, and a missing-field count. Removed constant, mostly missing, and highly correlated features using training data only.
(4) Compared 5 pipelines using 5-fold stratified cross-validation on X_train, with every fitted preprocessing step re-fitted inside each training fold. Results: HistGradientBoosting (shallow): PR-AUC 0.788, OOF cost 402 at t=0.050; HistGradientBoosting (deeper): PR-AUC 0.783, OOF cost 419 at t=0.030; Soft-vote ensemble (HGB + RF + LR): PR-AUC 0.735, OOF cost 433 at t=0.195; LogisticRegression: PR-AUC 0.636, OOF cost 597 at t=0.205; RandomForest: PR-AUC 0.429, OOF